# TruthLens — Train Image Forensics CNN from Scratch on Google Colab (GPU)

This standalone notebook trains the custom **TruthLens Image Forensic CNN** from scratch using a free Google Colab GPU (T4/V100) and genuine **GenImage** digital forensic benchmark data.

**Core Architecture:** 4-Block CNN with BatchNorm, Global Average Pooling, and Dropout (389,890 parameters).
**Pretraining:** NONE (100% trained from randomly initialised weights).
**Benchmark:** GenImage (ImageNet Real photos vs BigGAN, ADM, and unseen Midjourney synthetic images).

In [ ]:
# Step 1: Check GPU environment
!nvidia-smi
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))

In [ ]:
# Step 2: Install dependencies
!pip install -q pyarrow torchvision datasets scikit-learn matplotlib pillow

In [ ]:
# Step 3: Define Custom CNN Architecture from Scratch
import torch.nn as nn

class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )
    def forward(self, x):
        return self.b(x)

class TruthLensImageCNN(nn.Module):
    def __init__(self, num_classes=2, dropout=0.4):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128),
            ConvBlock(128, 256)
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x).view(x.size(0), -1)
        return self.classifier(x)

model = TruthLensImageCNN()
print('Total Parameters:', sum(p.numel() for p in model.parameters()))

In [ ]:
# Step 4: Download GenImage Arrow Shards from Hugging Face
from huggingface_hub import hf_hub_download
import os

os.makedirs('data/raw', exist_ok=True)
print('Downloading GenImage evaluation shards...')
biggan_0 = hf_hub_download(repo_id='nebula/GenImage-arrow', filename='data/test/BigGAN/data-00000-of-00002.arrow', repo_type='dataset')
biggan_1 = hf_hub_download(repo_id='nebula/GenImage-arrow', filename='data/test/BigGAN/data-00001-of-00002.arrow', repo_type='dataset')
adm_2    = hf_hub_download(repo_id='nebula/GenImage-arrow', filename='data/test/ADM/data-00002-of-00003.arrow', repo_type='dataset')
mj_17    = hf_hub_download(repo_id='nebula/GenImage-arrow', filename='data/test/Midjourney/data-00017-of-00019.arrow', repo_type='dataset')
print('Download completed successfully.')

In [ ]:
# Step 5: Extract Images and Build Splits (Train / Val / Test)
import pyarrow.ipc as ipc
import csv, io, hashlib, random
from PIL import Image

os.makedirs('data/images/real', exist_ok=True)
os.makedirs('data/images/ai_biggan', exist_ok=True)
os.makedirs('data/images/ai_adm', exist_ok=True)
os.makedirs('data/images/ai_mj', exist_ok=True)

def extract_shard(path, target_label, out_dir, prefix, max_count):
    entries = []
    with open(path, 'rb') as f:
        reader = ipc.RecordBatchStreamReader(f)
        for batch in reader:
            d = batch.to_pydict()
            for i in range(len(d['label'])):
                if d['label'][i] == target_label:
                    b = d['image'][i]
                    fn = f'{prefix}_{len(entries):06d}.jpg'
                    fp = os.path.join(out_dir, fn)
                    with open(fp, 'wb') as out_f:
                        out_f.write(b)
                    entries.append({'filepath': fp, 'label': target_label, 'generator': prefix})
                    if len(entries) >= max_count:
                        return entries
    return entries

print('Extracting real and synthetic images...')
reals = extract_shard(biggan_0, 0, 'data/images/real', 'real', 2400)
biggans = extract_shard(biggan_1, 1, 'data/images/ai_biggan', 'biggan', 1500)
adms = extract_shard(adm_2, 1, 'data/images/ai_adm', 'adm', 1500)
mjs = extract_shard(mj_17, 1, 'data/images/ai_mj', 'midjourney', 400)

# Allocate splits
train_set = reals[:1500] + biggans[:750] + adms[:750]
val_set = reals[1500:1900] + biggans[750:950] + adms[750:950]
test_set = reals[1900:2400] + biggans[950:1075] + adms[950:1075] + mjs[:250]

print(f'Train: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_set)}')

for name, sp in [('train', train_set), ('val', val_set), ('test', test_set)]:
    with open(f'data/{name}.csv', 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['filepath', 'label', 'generator'])
        w.writeheader()
        w.writerows(sp)

In [ ]:
# Step 6: Train on GPU (CUDA)
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Active Training Device:', device)

class ImgDataset(Dataset):
    def __init__(self, csv_p, tf):
        self.rows = list(csv.DictReader(open(csv_p)))
        self.tf = tf
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, idx):
        r = self.rows[idx]
        im = Image.open(r['filepath']).convert('RGB')
        return self.tf(im), int(r['label'])

tf_tr = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
tf_ev = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_loader = DataLoader(ImgDataset('data/train.csv', tf_tr), batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(ImgDataset('data/val.csv', tf_ev), batch_size=32, shuffle=False)
test_loader = DataLoader(ImgDataset('data/test.csv', tf_ev), batch_size=32, shuffle=False)

model = TruthLensImageCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print('Starting 5 training epochs on GPU...')
for epoch in range(1, 6):
    model.train()
    loss_sum, correct, total = 0, 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)
    print(f'Epoch {epoch}: Train Loss={loss_sum/total:.4f}, Train Acc={correct/total*100:.2f}%')

torch.save(model.state_dict(), 'truthlens_image_cnn_colab.pth')
print('Model checkpoint saved to truthlens_image_cnn_colab.pth')